# Wright Research Factor Rotation - Clean Research Hybrid Notebook

This notebook documents the clean research-oriented model used for the Wright Research factor rotation challenge.

It combines:

- an interpretable macro scorecard,
- macro-regime similarity,
- factor-rank persistence where appropriate,
- strict expanding walk-forward validation.

No public leaderboard calibration, hidden-label reconstruction, or row hardcoding is used in this notebook.

## Research Motivation

The model is inspired by factor timing literature:

- Time-series momentum: factor leadership may persist over short horizons.
- Regime changes: macro states change expected factor leadership.
- Forecast combination: simple weak signals can be more robust than one fitted model.
- Volatility/stress conditioning: factor attractiveness changes in stress regimes.

These papers guide the design, but no external data is used.

### Cell 1 - Import Libraries and Load Research Helpers

This cell configures paths and imports the clean research-hybrid script.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not ((PROJECT_ROOT / "work").exists() or (PROJECT_ROOT / "src").exists()):
    if (PROJECT_ROOT.parent / "work").exists() or (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

WORK_DIR = PROJECT_ROOT / "work"
if not WORK_DIR.exists():
    WORK_DIR = PROJECT_ROOT / "src"
OUT_DIR = PROJECT_ROOT / "notebook_outputs"
OUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(WORK_DIR))

DATA_DIR = Path(r"E:\wright_quant\data")
RANK_COLS = ["rank_momentum", "rank_quality", "rank_value"]

import paper_hybrid_strategy as hybrid

### Cell 2 - Load Competition Data

This cell loads train/test CSVs and checks the target rank columns.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print("train shape:", train.shape)
print("test shape:", test.shape)
train[["date", *RANK_COLS]].tail()

### Cell 3 - Build Walk-Forward Expert Predictions

This cell creates scorecard, regime-similarity, and persistence expert forecasts for validation.

In [ ]:
# Build walk-forward predictions for base experts: scorecard, regime similarity, previous-rank prior.
wf = hybrid.walk_forward_predictions(train)
search = hybrid.search_hybrids(wf)
search.head(10)[["name", "mean", "median", "hit", "y2020", "y2021", "y2022"]]

### Cell 4 - Generate Clean Research Hybrid Submission

This cell generates the clean 60% scorecard + 40% persistence research submission.

In [ ]:
# Clean research model used for submission_paper_hybrid_dma.csv
clean_weights = {"scorecard": 0.60, "regime": 0.00, "prev": 0.40, "const_m": 0.00}

out_clean = hybrid.generate_submission(train, test, clean_weights)
hybrid.validate(out_clean, test)
out_clean.to_csv(OUT_DIR / "submission_clean_research_hybrid_from_notebook.csv", index=False)
out_clean.head(12)

### Cell 5 - Generate Scorecard + Regime Variant

This cell generates a no-persistence regime-focused variant for private-period diversification.

In [ ]:
# Regime-focused variant: useful when we want less dependence on recursive prior ranks.
regime_weights = {"scorecard": 0.60, "regime": 0.40, "prev": 0.00, "const_m": 0.00}

out_regime = hybrid.generate_submission(train, test, regime_weights)
hybrid.validate(out_regime, test)
out_regime.to_csv(OUT_DIR / "submission_scorecard_regime_from_notebook.csv", index=False)
out_regime.head(12)

### Cell 6 - Compare Prediction Patterns

This cell summarizes how often each rank permutation appears in the generated submissions.

In [ ]:
# Quick comparison of generated rank patterns.
for name, df in {
    "clean_hybrid": out_clean,
    "scorecard_regime": out_regime,
}.items():
    print("\n", name)
    print(df[RANK_COLS].value_counts())

## Final Interpretation

The clean research hybrid had the best local walk-forward validation, while the more regime-focused variant is useful for private-period diversification.

